# From plans to shop drawings: end-to-end demonstration

This notebook explores one project, then runs the whole pipeline on it: extraction of the plan and of the shop drawings into JSON records (Appendix A of the brief), matching, comparison, and the PDF report.

Everything runs locally. The project PDFs are confidential and are not in the repository: put a project folder under `data/` (the plan PDF at its top, the shop drawings in folders under it) and set `PROJECT` below. Run the notebook from the root of the repository.

In [ ]:
import collections
import json
from pathlib import Path

import pandas as pd
import pymupdf

PROJECT = Path("data/CLP")

## 1. The data

A project is one structural plan (a PDF of many sheets) and the fabricator's shop drawings. The plan's sheets are numbered by series: S-100 foundations, S-300 beams, S-400 shear walls, S-500 columns, S-600 slabs.

In [ ]:
import extract_columns
import plan_columns

plan_pdf, column_pdfs = extract_columns.project_files([str(PROJECT)])
plan = pymupdf.open(plan_pdf)
titles = plan_columns.page_titles(plan)

SERIES = {"0": "general", "1": "S-100 foundations", "2": "S-200", "3": "S-300 beams", "4": "S-400 shear walls",
          "5": "S-500 columns", "6": "S-600 slabs"}
series = collections.Counter(SERIES.get(plan_columns.SHEET.search(title).group().replace(" ", "").lstrip("S-")[0], "other")
                             for title in titles.values())
print(f"{plan_pdf.name}: {len(plan)} sheets, {sum(len(page.get_text('words')) >= 20 for page in plan)} with a text layer")
pd.DataFrame(sorted(series.items()), columns=["series", "sheets"])

The shop drawings come in one folder per kind of element. A page either carries a text layer, which PyMuPDF reads exactly, or draws its text as outlines, which has to be read by OCR.

In [ ]:
rows = []
for folder in sorted({pdf.parent for pdf in PROJECT.rglob("*.pdf") if pdf.parent != PROJECT}):
    pages = [page for pdf in sorted(folder.glob("*.pdf")) for page in pymupdf.open(pdf)]
    rows.append({"folder": folder.name, "files": len(list(folder.glob("*.pdf"))), "pages": len(pages),
                 "pages with a text layer": sum(len(page.get_text("words")) >= 20 for page in pages)})
pd.DataFrame(rows)

## 2. Extracting the plan

On a column sheet every column is a black shape, tied by a leader line to a tag that gives its reinforcement: the vertical bars (`ARM.`) and the ties (`LIG.`). The tag carries no name, so the column is named after the two grid lines it stands on, as fabricators do.

`plan_columns.extract` finds the tags, follows each leader to its column, reads the grid, and returns one record per column.

In [ ]:
plan_records, plan_report, links = plan_columns.extract(plan_pdf)
print(len(plan_records), "column records")
pd.DataFrame(plan_report).T[["tags", "columns", "by leader", "grid lines", "named between lines", "unnamed"]].rename_axis("page")

A record, in the format of Appendix A. `x`, `y` are the centre of the tag in PDF points from the top-left corner of the page.

In [ ]:
print(json.dumps(plan_records[0].model_dump(), indent=1, ensure_ascii=False))

## 3. Extracting the shop drawings and writing the JSON

Fabricators draw columns in two ways, and both are read:

- a storey table, one table column per building column (`atelier_columns.py`);
- one detail per column, or per group of identical columns, side by side (`atelier_details.py`). These drawings often have no text layer and are then read by OCR, once: the reading is kept in `out/ocr_cache/`.

`extract_columns.run` runs the plan and the shop-drawing extraction, reconciles the names of columns that stand between grid lines, and writes the two JSON files. It also writes check PDFs in `out/`, the source drawings with what was extracted drawn on top.

In [ ]:
plan_json, atelier_json = extract_columns.run([str(PROJECT)])

In [ ]:
plan_data = json.loads(plan_json.read_text(encoding="utf8"))
atelier_data = json.loads(atelier_json.read_text(encoding="utf8"))
print(json.dumps(atelier_data[0], indent=1, ensure_ascii=False))

Every record is validated against the schema of Appendix A when it is written. The same check, on the files:

In [ ]:
import schema

records = [schema.Element(**record) for record in plan_data + atelier_data]
print(len(records), "records comply with the schema")
pd.DataFrame(collections.Counter((r.source, r.feuillet) for r in records).items(), columns=["key", "records"]).assign(
    source=lambda t: t.key.str[0], sheet=lambda t: t.key.str[1]).pivot(index="sheet", columns="source", values="records")

## 4. Matching and comparison

A plan record and a shop-drawing record describe the same column when they share the plan sheet and the element name. What is left alone is *missing from the shop drawings* or *added in the shop drawings*.

For a pair, the bars are aligned by kind (counted bars, spaced bars) and diameter, not by their position in the list: the plan summarises the reinforcement while the shop drawing lists every line. A field is compared only when the plan gives it. A tie zone tighter than the plan's spacing complies; a wider one does not.

In [ ]:
from l2c.appariement import Apparieur
from l2c.comparaison import Comparateur
from l2c.modeles import Element

elements = [Element.depuis_dict(record) for record in plan_data + atelier_data]
verdicts = Comparateur().comparer_paires(Apparieur(elements).apparier())

sheet_of = lambda verdict: (verdict.paire.plan or verdict.paire.atelier).feuillet
table = pd.crosstab(pd.Series([sheet_of(v) for v in verdicts], name="sheet"), pd.Series([v.statut for v in verdicts], name="verdict"))
table.loc["all"] = table.sum()
table

In [ ]:
ax = table.drop("all").plot.bar(stacked=True, figsize=(8, 3), color={"conforme": "#4c9f70", "non_conforme": "#d1495b", "manquant": "#edae49", "ajoute": "#00798c"})
ax.set_ylabel("columns")
ax.set_title("Verdicts per plan sheet");

The discrepancies of the non-compliant elements, with where to find each element on the plan:

In [ ]:
rows = []
for verdict in verdicts:
    for ecart in verdict.ecarts:
        plan_card = verdict.paire.plan
        value = lambda v: "-" if v is None else getattr(v, "diametre", v)
        rows.append({"sheet": plan_card.feuillet, "element": plan_card.element, "field": ecart.champ,
                     "plan": value(ecart.valeur_plan), "shop drawing": value(ecart.valeur_atelier),
                     "plan page": plan_card.page, "x": plan_card.x, "y": plan_card.y})
pd.DataFrame(rows)

## 5. The report

The PDF report gives, for each plan sheet, the number of conformities and of non-conformities, then the detail of every element to flag, with the differing values in red and the position of the element on the plan and on the shop drawing.

In [ ]:
from l2c.rapport import Rapport

report_path = Path("out") / f"{PROJECT.name}_rapport.pdf"
Rapport(verdicts).ecrire_pdf(report_path)
print(report_path, "-", len(pymupdf.open(report_path)), "pages")

The whole of sections 2 to 5 is one command:

```
python main.py data/CLP
```

## 6. Reading drawings that have no text layer

Pages without a text layer are rendered at 300 dpi, cut into overlapping tiles and read by OCR twice, as displayed and turned a quarter turn for vertical text; duplicate readings are then removed (`start.py`). The OCR is PaddleOCR's pretrained PP-OCRv6 model run through RapidOCR. No model was trained.

How reliable is it? `ocr_check.py` scores the OCR on pages that *do* have a text layer, using that layer as the answer key. On 41 such pages (36,254 numbers):

| | read exactly |
|---|---|
| all numbers | 93.6% |
| column drawings | 99% |
| slab and foundation sheets | about 90% |
| bar diameters | 98.5% |
| vertical text | 90.5% |

Two rules came out of checking the OCR by hand on drawings without a text layer (`ocr_review.py`). A longer reading of a text wins over a shorter one with a better score, which had been losing the number after bold titles. And a lone "1" is trusted only if its ink has the shape of the digit: a stem with a hook on its top left and nothing on its top right. Arrows and line ends, which OCR reads as "1", fail the test and are marked doubtful instead of being used:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import start

def glyph(*strokes):
    gray = np.full((40, 30), 255.0)
    for top, bottom, left, right in strokes:
        gray[top:bottom, left:right] = 0
    return gray

stem = (4, 36, 16, 20)
shapes = {"digit 1": glyph(stem, (6, 10, 8, 16)), "bare line": glyph(stem), "arrow on a line": glyph(stem, (6, 14, 8, 28), (36, 38, 2, 28))}
figure, axes = plt.subplots(1, 3, figsize=(6, 2.6))
for ax, (name, gray) in zip(axes, shapes.items()):
    ax.imshow(gray, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{name}\n{'a 1' if start.looks_like_one(gray) else 'not a 1'}", fontsize=9)
    ax.axis("off")

## 7. Limits

- Only columns (S-500) are extracted; foundations, beams, shear walls and slabs are not.
- A sheet of details is filed under a plan sheet from the storey in its file name (`...NIV-2@3`); without it the drawing is left out.
- A column is matched by its name. When the two names differ (a column between grid lines, a grid line the plan reader does not know, a name OCR misread), it shows as one missing and one added element.
- The flagged quantities were not checked against the drawings one by one: some are real, some come from a line that was not read or was read into the wrong detail.
- OCR misses about 4% of numbers and misreads about 2%; these become false alarms or, more rarely, hide a discrepancy.